# Stokes Equations — FEniCSx Demo

Solve the Stokes equations on the unit square (lid-driven cavity):

$$-\nu \Delta u + \nabla p = f$$
$$\nabla \cdot u = 0$$

with no-slip BCs on walls and a driven lid (u = (1,0) on top boundary).
Uses Taylor-Hood elements: P2 for velocity, P1 for pressure.

In [1]:
from mpi4py import MPI
import numpy as np
from dolfinx import mesh, fem, default_scalar_type
from dolfinx.fem import functionspace, Function, dirichletbc, locate_dofs_topological
from dolfinx.fem.petsc import LinearProblem
from dolfinx.mesh import locate_entities_boundary
from ufl import (TestFunctions, TrialFunctions, dx, grad, div,
                 inner, nabla_grad, MixedFunctionSpace)
import basix.ufl

In [2]:
domain = mesh.create_unit_square(MPI.COMM_WORLD, 16, 16)

P2 = basix.ufl.element("Lagrange", domain.topology.cell_name(), 2, shape=(domain.geometry.dim,))
P1 = basix.ufl.element("Lagrange", domain.topology.cell_name(), 1)
TH = basix.ufl.mixed_element([P2, P1])
W = functionspace(domain, TH)

print(f"Total dofs: {W.dofmap.index_map.size_global * W.dofmap.index_map_bs}")

Total dofs: 2467


In [3]:
W0 = W.sub(0)  # velocity subspace
W1 = W.sub(1)  # pressure subspace

# No-slip on all walls
def noslip_boundary(x):
    return np.logical_or.reduce([
        np.isclose(x[0], 0),   # left
        np.isclose(x[0], 1),   # right
        np.isclose(x[1], 0)    # bottom
    ])

# Driven lid: u = (1, 0) on top
def lid_boundary(x):
    return np.isclose(x[1], 1)

V, _ = W.sub(0).collapse()

# No-slip BC
noslip_dofs = locate_dofs_topological(
    (W.sub(0), V), domain.topology.dim - 1,
    locate_entities_boundary(domain, domain.topology.dim - 1, noslip_boundary))
u_noslip = Function(V)
u_noslip.x.array[:] = 0
bc_noslip = dirichletbc(u_noslip, noslip_dofs, W.sub(0))

# Lid BC
lid_dofs = locate_dofs_topological(
    (W.sub(0), V), domain.topology.dim - 1,
    locate_entities_boundary(domain, domain.topology.dim - 1, lid_boundary))
u_lid = Function(V)
u_lid.interpolate(lambda x: np.vstack([np.ones(x.shape[1]), np.zeros(x.shape[1])]))
bc_lid = dirichletbc(u_lid, lid_dofs, W.sub(0))

bcs = [bc_noslip, bc_lid]

In [4]:
(u, p) = TrialFunctions(W)
(v, q) = TestFunctions(W)

nu = fem.Constant(domain, default_scalar_type(0.01))
f = fem.Constant(domain, (default_scalar_type(0), default_scalar_type(0)))

a = (nu * inner(grad(u), grad(v)) - p * div(v) + div(u) * q) * dx
L = inner(f, v) * dx

problem = LinearProblem(a, L, bcs=bcs, petsc_options_prefix="stokes_",
                        petsc_options={"ksp_type": "preonly", "pc_type": "lu",
                                       "pc_factor_mat_solver_type": "mumps"})
wh = problem.solve()
uh, ph = wh.split()
print("Solve complete.")

Solve complete.


In [5]:
import pyvista as pv
from dolfinx.plot import vtk_mesh

pv.set_jupyter_backend("static")

V_vis, _ = W.sub(0).collapse()
u_vis = Function(V_vis)
u_vis.x.array[:] = uh.x.array

topology, cell_types, geometry = vtk_mesh(V_vis)
grid = pv.UnstructuredGrid(topology, cell_types, geometry)

# Plot velocity magnitude
u_array = u_vis.x.array.reshape(-1, domain.geometry.dim)
grid.point_data["u_mag"] = np.linalg.norm(u_array, axis=1)

plotter = pv.Plotter()
plotter.add_mesh(grid, scalars="u_mag", cmap="viridis")
plotter.view_xy()
plotter.show()

ValueError: could not broadcast input array from shape (2467,) into shape (2178,)